In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.feature_selection import SelectKBest, f_regression

from sklearn.metrics import root_mean_squared_error, r2_score



X, y = load_diabetes(as_frame=True, scaled=False, return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

baseline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

baseline.fit(X_train, y_train)
pred_base = baseline.predict(X_test)

rmse_base = root_mean_squared_error(y_test, pred_base)
r2_base = r2_score(y_test, pred_base)

print("BASELINE")
print("RMSE:", round(rmse_base, 2))
print("R2  :", round(r2_base, 3))

BASELINE
RMSE: 53.32
R2  : 0.486


In [2]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print("Antal originalfeatures:", X_train.shape[1])
print("Antal features efter PolynomialFeatures(degree=2):", X_train_poly.shape[1])

Antal originalfeatures: 10
Antal features efter PolynomialFeatures(degree=2): 65


In [ ]:
pipe_fs = Pipeline(steps=[
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scaler", StandardScaler()),
    ("select", SelectKBest(score_func=f_regression)),
    ("model", Ridge())
])

param_grid = {
    "poly__degree": [1, 2],
    "select__k": [5, 10, 20, 30, 40, 65],
    "model__alpha": [0.1, 1.0, 10.0, 100.0]
}


cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe_fs,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Bästa params:", grid.best_params_)
print("Bästa CV RMSE:", round(-grid.best_score_, 2))

Bästa params: {'model__alpha': 100.0, 'poly__degree': 2, 'select__k': 65}
Bästa CV RMSE: 56.04
